# OntoLayer on Colab
Define business concepts in YAML, build a knowledge graph, and let an agent query it through **MCP**.

Run the cells top to bottom. Steps 1-4 need no API key.

## 1. Install

In [ ]:
import os, sys, subprocess

# Option A: clone your GitHub repo (edit the URL). Option B: upload ontolayer.zip when prompted.
REPO_URL = "https://github.com/YOUR_USER/ontolayer.git"

if not os.path.exists("/content/ontolayer"):
    if "YOUR_USER" not in REPO_URL:
        subprocess.run(["git", "clone", REPO_URL, "/content/ontolayer"], check=True)
    else:
        from google.colab import files
        up = files.upload()  # choose ontolayer.zip
        subprocess.run(["unzip", "-q", next(iter(up)), "-d", "/content"], check=True)

%cd /content/ontolayer
!pip install -q -e .

## 2. Load the ontology and build the knowledge graph

In [ ]:
import json
from ontolayer import load_ontology, KnowledgeGraph, SemanticLayer

path = "examples/retail/ontology.yaml"
onto = load_ontology(path)
kg = KnowledgeGraph.build(onto, "examples/retail")
layer = SemanticLayer(kg)
print(json.dumps(kg.stats(), indent=2))

## 3. Ask the semantic layer directly (plain Python)

In [ ]:
print(json.dumps(layer.check_answerable("What was revenue by country for business customers?"), indent=2))
print(json.dumps(layer.compute_metric("Revenue", group_by="Customer.country")["groups"], indent=2))

## 4. Use it as an agent would: over MCP
`ask` starts the MCP server as a subprocess and talks to it through the protocol. `mode="offline"` uses a rule-based planner (no key needed).
Colab supports top-level `await`, so we use it instead of `asyncio.run`.

In [ ]:
from ontolayer.agent import ask

await ask(path, "What was revenue by country for business customers?", mode="offline")
print()
await ask(path, "How many orders were cancelled?", mode="offline")
print()
await ask(path, "What was the return on ad spend?", mode="offline")   # not in the ontology -> refuses

## 5. Same thing with a real LLM (any OpenAI-compatible endpoint)
Pick the cheapest option for you. In Colab, open the key icon (Secrets) and add `LLM_API_KEY`, and turn on notebook access.
Edit `LLM_BASE_URL` and `LLM_MODEL` for your provider (check its docs: model names and free-tier limits change).

In [ ]:
from google.colab import userdata
from ontolayer.agent import LLMConfig

cfg = LLMConfig(
    base_url="https://generativelanguage.googleapis.com/v1beta/openai",   # example: Gemini's OpenAI-compatible endpoint
    api_key=userdata.get("LLM_API_KEY"),
    model="gemini-2.5-flash",                                              # change if your provider renamed it
)

result = await ask(path, "Which product category sells the most units, and how many?", mode="llm", cfg=cfg)

## 6. Your own data
Put CSVs in a folder and draft an ontology from them, then review and edit it:
```python
!ontolayer suggest my_csv_folder -o my_csv_folder/ontology.yaml
```